# Workflow Optimization

Optimization helps you systematically improve your agent's performance by tuning prompts and parameters. This tutorial covers NAT's optimization capabilities.

## What You'll Learn

1. Understanding optimization types
2. Configuring numeric optimization (hyperparameters)
3. Configuring prompt optimization
4. Running optimization via SDK
5. Running optimization via CLI
6. Applying optimized configurations

## Why Optimize?

- **Improve accuracy** - Find the best prompts and parameters
- **Reduce costs** - Optimize for efficiency
- **Automate tuning** - Replace manual trial-and-error
- **Data-driven decisions** - Use metrics to guide improvements


In [1]:
import sys
from pathlib import Path

# Fix for asyncio in Jupyter notebooks
import nest_asyncio

nest_asyncio.apply()

# Setup
module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

print("✅ Environment configured")


✅ Environment configured


## Optimization Types

NAT supports two types of optimization:

### 1. Numeric Optimization (Optuna-based)
Uses Optuna to optimize numerical/hyperparameters:
- `temperature` - LLM randomness (0.0-1.0)
- `max_tokens` - Response length
- `top_p` - Nucleus sampling parameter

Configuration: `NumericOptimizationConfig`

### 2. Prompt Optimization (Genetic Algorithm)
Uses a Genetic Algorithm to evolve prompts:
- System prompts
- Additional instructions
- Few-shot examples

Configuration: `PromptGAOptimizationConfig`


## Step 1: Create a Workflow to Optimize


### Configuring Optimizable Parameters

Components like LLMs support optimization through two fields:

1. **`optimizable_params`** - A list of field names to optimize:
   ```python
   optimizable_params=["temperature", "top_p", "max_tokens"]
   ```

2. **`search_space`** - A dictionary defining the search range for each parameter:
   ```python
   search_space={
       "temperature": SearchSpace(low=0.1, high=0.9, step=0.2),
       "model_name": SearchSpace(values=["model-a", "model-b"]),
   }
   ```

**SearchSpace options:**
- **Continuous**: `SearchSpace(low=0.1, high=0.9, step=0.2)` - Range from 0.1 to 0.9 in 0.2 increments
- **Categorical**: `SearchSpace(values=["a", "b", "c"])` - Choose from specific values
- **Default ranges**: Many LLM parameters have built-in default search spaces, so you can just list them in `optimizable_params` without specifying `search_space`


In [2]:
import json

from nat.agent.sdk import NatReActAgent
from nat.llm.sdk import NimLLM
from nat.tool.sdk import CurrentTimeTool
from nat.utils.sdk.nat_optimizer import SearchSpace
from nat.utils.sdk.nat_workflow import NatWorkflow

# Create LLM with optimizable parameters
# Use `optimizable_params` to specify which fields to optimize
# Use `search_space` to define/override the search ranges
llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.5,      # Starting value
    max_tokens=1024,
    name="nim_llm",
    # Specify which parameters to optimize
    optimizable_params=["temperature", "top_p"],
    # Define custom search ranges (optional - uses defaults if not specified)
    search_space={
        "temperature": SearchSpace(low=0.1, high=0.8, step=0.1),
        "top_p": SearchSpace(low=0.7, high=1.0, step=0.1),
    },
)

# Create tools
time_tool = CurrentTimeTool(name="current_time")

try:
    from nat_simple_calculator.sdk import CalculatorToolGroup
    calculator = CalculatorToolGroup(name="calculator")
    tools = [time_tool, calculator]
except ImportError:
    tools = [time_tool]

# Create agent with system prompt (we'll optimize this too)
agent = NatReActAgent(
    tools=tools,
    llm=llm,
    verbose=True,
    additional_instructions="Be concise in your responses.",  # Will optimize
)

workflow = NatWorkflow(entrypoint=agent)
print("✅ Workflow created")


/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


✅ Workflow created


## Step 2: Create Optimization Dataset

The optimizer needs a dataset to evaluate performance:


In [3]:
# Create optimization dataset (using correct field names: id, question, answer)
opt_data = [
    {"id": "add_001", "question": "What is 2 + 2?", "answer": "The answer is 4."},
    {"id": "mul_001", "question": "What is 10 * 5?", "answer": "The answer is 50."},
    {"id": "div_001", "question": "What is 100 / 4?", "answer": "The answer is 25."},
    {"id": "sub_001", "question": "What is 15 - 7?", "answer": "The answer is 8."},
]

# Save dataset
data_dir = Path("./data")
data_dir.mkdir(parents=True, exist_ok=True)

dataset_path = data_dir / "opt_dataset.json"
with open(dataset_path, "w") as f:
    json.dump(opt_data, f, indent=2)

print(f"📊 Dataset saved to: {dataset_path}")


📊 Dataset saved to: data/opt_dataset.json


## Step 3: Configure the Optimizer

The optimizer requires:
1. **Evaluation metrics** (`eval_metrics`) - Dictionary of metrics to optimize with their direction and weights
2. **Numeric optimization** (`NumericOptimizationConfig`) - Optuna-based hyperparameter tuning
3. **Prompt optimization** (`PromptGAOptimizationConfig`) - GA-based prompt evolution (optional)


In [4]:
from nat.eval.sdk import RagasEvaluator
from nat.utils.sdk.nat_evaluation import EvalDatasetJsonConfig
from nat.utils.sdk.nat_evaluation import NatEvaluation
from nat.utils.sdk.nat_optimizer import NatOptimizer
from nat.utils.sdk.nat_optimizer import NumericOptimizationConfig
from nat.utils.sdk.nat_optimizer import OptimizerMetric
from nat.utils.sdk.nat_optimizer import PromptGAOptimizationConfig

# Create evaluator for optimization metric
accuracy_eval = RagasEvaluator(
    llm=llm,
    metric="AnswerAccuracy",
    name="accuracy",
)

# Configure evaluation (required for optimization)
evaluation = NatEvaluation(
    output_dir=Path("./opt_results/eval"),
    dataset=EvalDatasetJsonConfig(file_path=dataset_path),
    evaluators=[accuracy_eval],
)

workflow.add_evaluator(evaluation)
print("✅ Evaluator configured")


✅ Evaluator configured


In [5]:
# Configure optimizer
# Note: The optimizer uses Optuna for numeric optimization and a Genetic Algorithm for prompt optimization
optimizer = NatOptimizer(
    output_path=Path("./opt_results"),

    # What metrics to optimize (dict with evaluator_name as key)
    eval_metrics={
        "accuracy": OptimizerMetric(
            evaluator_name="accuracy",   # Must match evaluator name
            direction="maximize",        # "maximize" or "minimize"
            weight=1.0,                  # Weight for multi-objective optimization
        )
    },

    # Numeric optimization configuration (Optuna-based)
    numeric=NumericOptimizationConfig(
        enabled=True,
        n_trials=5,  # Number of optimization trials
    ),

    # Prompt optimization configuration (GA-based) - optional
    prompt=PromptGAOptimizationConfig(
        enabled=False,  # Disabled by default - enable for prompt tuning
        ga_population_size=8,
        ga_generations=5,
    ),

    # Number of evaluations per parameter set
    reps_per_param_set=1,

    # Target value to stop early (optional)
    target=0.95,
)

workflow.add_optimizer(optimizer)
print("✅ Optimizer configured")


✅ Optimizer configured


## Step 4: Save the Configuration


In [6]:
# Save configuration
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "opt_workflow.yaml"
workflow.save_to_config_file(config_path)

print(f"📄 Configuration saved to: {config_path}")
print("\n" + "=" * 60)
print("GENERATED CONFIGURATION WITH OPTIMIZER:")
print("=" * 60 + "\n")

with open(config_path) as f:
    print(f.read())


📄 Configuration saved to: configs/opt_workflow.yaml

GENERATED CONFIGURATION WITH OPTIMIZER:

functions:
  current_time:
    _type: current_datetime

function_groups:
  calculator:
    _type: calculator

llms:
  nim_llm:
    _type: nim
    optimizable_params:
    - temperature
    - top_p
    model: meta/llama-3.3-70b-instruct
    max_tokens: 1024
    temperature: 0.5

optimizer:
  output_path: opt_results
  eval_metrics:
    accuracy:
      evaluator_name: accuracy
      direction: maximize
      weight: 1.0
  reps_per_param_set: 1
  target: 0.95
  multi_objective_combination_mode: harmonic
  numeric:
    enabled: true
    n_trials: 5
  prompt:
    enabled: false
    ga_population_size: 8
    ga_generations: 5

workflow:
  _type: react_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - current_time
  - calculator
  additional_instructions: Be concise in your responses.

eval:
  general:
    output_dir: opt_results/eval
    dataset:
      _type: json
      file_path: data/opt_

## Step 5: Run Optimization

### Via Python SDK


In [7]:
# Run optimization (uncomment to execute)
await workflow.optimize()
print("✅ Optimization complete! Check ./opt_results for results")


The Optimizer feature is experimental and the API may change in future releases. Future versions may introduce breaking changes without notice. Function: nat.profiler.parameter_optimization.optimizer_runtime.optimize_config
The Optimizer feature is experimental and the API may change in future releases. Future versions may introduce breaking changes without notice. Function: nat.profiler.parameter_optimization.parameter_optimizer.optimize_parameters
[I 2025-12-29 19:21:21,967] A new study created in memory with name: no-name-4741f75b-f083-4ef0-84d4-99dcba0cce38
Evaluating Ragas nv_accuracy: 100%|██████████| 4/4 [00:01<00:00,  2.94it/s]
[I 2025-12-29 19:21:26,441] Trial 0 finished with value: 1.0 and parameters: {'llms.nim_llm.temperature': 0.1, 'llms.nim_llm.top_p': 0.7}. Best is trial 0 with value: 1.0.
Evaluating Ragas nv_accuracy: 100%|██████████| 4/4 [00:01<00:00,  2.71it/s]
[I 2025-12-29 19:21:29,676] Trial 1 finished with value: 1.0 and parameters: {'llms.nim_llm.temperature': 0.

✅ Optimization complete! Check ./opt_results for results


## Step 6: Visualize Optimization Results

Let's visualize what the optimizer discovered:


In [8]:
from utils.visualization import visualize_optimization

# Visualize the optimization results and get best parameters
best_params = visualize_optimization("opt_results")


📊 Loading optimization results...
   Found 5 trials

🏆 BEST CONFIGURATION FOUND

📈 Trial #0 achieved the best score!

   Accuracy Score: 1.0000
   Duration: 4.47 seconds

📋 Optimal Parameters:
----------------------------------------
   temperature: 0.1
   top_p: 0.7
----------------------------------------

💡 Use these parameters in your workflow for optimal performance!

📈 Creating visualization...
📊 Visualization saved to: opt_results/optimization_summary.png


/Users/spastoriza/Documents/Programming/public/nat-fork/examples/notebooks/sdk/utils/visualization.py:845: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# View the optimized configuration file
with open("opt_results/optimized_config.yml") as f:
    import yaml
    optimized = yaml.safe_load(f)

print("🏆 Best LLM Parameters Found:")
print(f"   temperature: {optimized['llms']['nim_llm']['temperature']}")
print(f"   top_p: {optimized['llms']['nim_llm']['top_p']}")

print("\n" + "=" * 60)
print("📄 FULL OPTIMIZED CONFIGURATION:")
print("=" * 60 + "\n")

with open("opt_results/optimized_config.yml") as f:
    print(f.read())


🏆 Best LLM Parameters Found:
   temperature: 0.1
   top_p: 0.7

📄 FULL OPTIMIZED CONFIGURATION:

functions:
  current_time:
    _type: current_datetime
function_groups:
  calculator:
    _type: calculator
llms:
  nim_llm:
    _type: nim
    model: meta/llama-3.3-70b-instruct
    max_tokens: 1024
    temperature: 0.1
    top_p: 0.7
workflow:
  _type: react_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - current_time
  - calculator
  additional_instructions: Be concise in your responses.
eval:
  general:
    output_dir: opt_results/eval
    dataset:
      _type: json
      file_path: data/opt_dataset.json
  evaluators:
    accuracy:
      _type: ragas
      llm_name: nim_llm
      metric: AnswerAccuracy



### Via CLI

```bash
# Run optimization
nat optimize --config_file configs/opt_workflow.yaml

# With custom dataset
nat optimize --config_file configs/opt_workflow.yaml \
    --dataset data/opt_dataset.json

# Override output path
nat optimize --config_file configs/opt_workflow.yaml \
    --override optimizer.output_path=./custom_results
```


## Understanding Optimization Results

After optimization, you'll find:

```
opt_results/
├── optimized_config.yml         # Best configuration found
├── trials_dataframe_params.csv  # All trial results with parameters and scores
├── config_numeric_trial_0.yml   # Config used for trial 0
├── config_numeric_trial_1.yml   # Config used for trial 1
├── ...                          # One config file per trial
├── eval/                        # Evaluation results from the last trial
│   ├── accuracy_output.json     # Accuracy scores for each test case
│   └── workflow_output.json     # Raw workflow outputs with intermediate steps
└── plots/                       # Pareto front visualizations (if generated)
```

### Trials DataFrame (`trials_dataframe_params.csv`)

The CSV contains detailed information about each optimization trial:

| Column | Description |
|--------|-------------|
| `number` | Trial index (0, 1, 2, ...) |
| `value` | Optimization score achieved |
| `datetime_start` | When the trial started |
| `datetime_complete` | When the trial completed |
| `duration` | How long the trial took |
| `params_*` | Parameter values tested (e.g., `params_llms.nim_llm.temperature`) |
| `rep_scores` | Scores for each repetition |
| `state` | Trial state (COMPLETE, FAILED, etc.) |
| `pareto_optimal` | Whether this trial is on the Pareto front |


## Summary

In this tutorial, you learned:

✅ Understanding numeric and prompt optimization  
✅ Configuring `NatOptimizer` with metrics and parameters  
✅ Defining search spaces for optimization  
✅ Running optimization via SDK and CLI  
✅ Understanding optimization results  

## Next Steps

- **[13_observability.ipynb](./13_observability.ipynb)** - Add tracing and monitoring
- **[09_configuration_guide.ipynb](./09_configuration_guide.ipynb)** - Deep dive into YAML configuration
